# Lines相关

## Rename L3D++_data，从 img_name 命名改为 img_id

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import random
import networkx as nx
import matplotlib.pyplot as plt

from datasets.dataset_reader import load_sparse_model


def main_colmap():
    ####################################### 参数 #######################################
    sparse_model_path = r"/home/rylynn/Pictures/LinesDetection_Workspace/datasets/Dublin_block3/sparse_txt/"
    line2d_path = r"/home/rylynn/Pictures/LinesDetection_Workspace/output/dublin_block3_md_resize4/L3D++_data/"
    camerasInfo, _ = load_sparse_model(sparse_model_path, image_scale=1)
    for cam_id, cam_dict in enumerate(camerasInfo):
        width = int(cam_dict['width'])
        height = int(cam_dict['height'])
        img_name = cam_dict['img_name'].split('/')[-1]
        line2d_filename1 = f'segments_L3D++_{img_name}_{width}x{height}.bin'
        cam_id_new = cam_id+1 # cam_id 从0开始，img_id从1开始
        line2d_filename2 = f'segments_L3D++_{cam_id_new}_{width}x{height}.bin'
        old_path = os.path.join(line2d_path, line2d_filename1)
        new_path = os.path.join(line2d_path, line2d_filename2)
        os.rename(old_path, new_path)
        print(f"{img_name}更名成功")


if __name__ == "__main__":
    main_colmap()

## 从L3D++_data中选择长度前3000的线段

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import random
import networkx as nx
import matplotlib.pyplot as plt
import cv2

from datasets.dataset_reader import (
    load_sparse_model,
    match_pair,
    read_depth
)
from datasets.line3dpp_loader import parse_line_segments, parse_lines3dpp, save_segments_l3dpp
from utils.visualize import viz_lines2D2

if __name__ == "__main__":
    ####################################### 参数 #######################################
    workspace = r"/home/rylynn/Pictures/LinesDetection_Workspace/datasets/group611_800z"

    ####################################### 路径 #######################################
    sparse_model_path = os.path.join(workspace, 'sparse_txt')
    images_path = os.path.join(workspace, 'images')
    output_path = '/home/rylynn/Pictures/LinesDetection_Workspace/datasets/group611_800z/sparse_txt_geo/'
    lsd_lines_path = os.path.join(output_path, 'Line3D++_H')
    len3000_path = os.path.join(output_path, 'Line3D++')
    os.makedirs(len3000_path, exist_ok=True)
    viz_path = os.path.join(len3000_path, 'viz_lsd_len3000')
    os.makedirs(viz_path, exist_ok=True)
    

    # 0. 数据准备：读取稀疏模型，读取LSD检测所有线段
    camerasInfo, points_in_images = load_sparse_model(sparse_model_path, image_scale=1)
    print(f"[INFO] Loaded {len(camerasInfo)} images.")

    # 1. 取长度前3000的线段存储为Line3D++的输入
    topk_indices_all = {}
    for img_id, cam_dict in tqdm(enumerate(camerasInfo), total=len(camerasInfo), desc="Processing lines"):
        cam_dict = camerasInfo[img_id]
        width = int(cam_dict['width'])
        height = int(cam_dict['height'])
        img_name = cam_dict['img_name'].split('/')[-1]

        # 0205:预先把空的文件删除了，所以这里加个判断
        filename = 'segments_L3D++_'+ str(img_id+1) + '_' + str(width) + 'x' + str(height) + '.bin'
        filepath = os.path.join(lsd_lines_path, 'L3D++_data', filename)
        if not os.path.exists(filepath):
            print(f"[WARNING] File {filepath} does not exist. Skipping.")
            continue

        lines = parse_line_segments(lsd_lines_path, img_id+1, width, height)[:, [1,0,3,2]]


        # 取长度前3000的线段
        lines_lengths = ((lines[:, 0]-lines[:, 2])**2 + (lines[:, 1]-lines[:, 3])**2)**0.5

        # 对长度进行降序排序，获取对应的原始索引
        # argsort 返回的是：原本在哪个位置的元素现在排在这里
        sorted_indices = np.argsort(-lines_lengths)
        # 取前 3000 个索引
        top_k = min(3000, lines.shape[0])  # 防止 lines 总数不足 3000
        topk_indices = sorted_indices[:top_k]
        # 根据索引提取对应的线段
        lines_topk = lines[topk_indices]
        topk_indices_all[img_id] = topk_indices.tolist()
 
        save_segments_l3dpp(lines_topk, len3000_path, img_id+1, width, height)
        #img = cv2.imread(os.path.join(images_path, cam_dict['img_name']+'.png'), 0)
        #viz_lines2D2(img, lines_topk[:, [1,0,3,2]].reshape(-1,2,2), viz_path, f"{os.path.splitext(img_name)[0]}")
    
    # 保存top3000线段的索引
    #with open(os.path.join(len3000_path, 'top3000_indices.json'), 'w') as f:
        #json.dump(topk_indices_all, f)


## convert xml(Line3D++) to bin(ISPRS Journal2024)

In [ ]:
import os
import json
import numpy as np
from tqdm import tqdm
import random
import networkx as nx
import matplotlib.pyplot as plt
import struct
import re
import math

from datasets.dataset_reader import (
    load_sparse_model,
    match_pair,
    read_depth
)
from datasets.line3dpp_loader import parse_line_segments, parse_lines3dpp, save_segments_l3dpp
from utils.visualize import viz_lines2D2


def convert_xml_to_opencv_bin(input_xml_path, output_bin_path):
    """
    读取XML文件，转换为C++ OpenCV可读的二进制文件
    每行包含7个浮点数: x1, y1, x2, y2, cx, cy, length
    """
    with open(input_xml_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 提取线段端点坐标
    pattern = re.compile(
        r'<x>([0-9eE\+\-\.]+)</x>\s*'
        r'<y>([0-9eE\+\-\.]+)</y>\s*'
        r'<z>([0-9eE\+\-\.]+)</z>\s*'
        r'<w>([0-9eE\+\-\.]+)</w>'
    )

    matches = pattern.findall(content)
    filename = os.path.basename(input_xml_path)

    if not matches:
        print(f"[跳过] {filename}: 未在文件中找到线段数据！")
        return False

    num_lines = len(matches)
    
    # 设置矩阵格式：N行 x 7列 的单通道矩阵 (CV_32F, type=5)
    rows = num_lines
    cols = 7  # 每行7个数字
    mat_type = 5  # CV_32F (单通道浮点)

    with open(output_bin_path, 'wb') as f:
        # 写入头部 (3个整型: rows, cols, type)，使用小端序 '<i'
        f.write(struct.pack('<i', rows))
        f.write(struct.pack('<i', cols))
        f.write(struct.pack('<i', mat_type))

        # 写入矩阵数据：每行7个浮点数
        for match in matches:
            x1, y1, x2, y2 = map(float, match)
            
            # 计算中心点和长度
            cx = (x1 + x2) / 2.0
            cy = (y1 + y2) / 2.0
            length = math.sqrt((x1 - x2) * (x1 - x2) + (y1 - y2) * (y1 - y2))
            
            # 按顺序写入7个浮点数
            f.write(struct.pack('<f', x1))
            f.write(struct.pack('<f', y1))
            f.write(struct.pack('<f', x2))
            f.write(struct.pack('<f', y2))
            f.write(struct.pack('<f', cx))
            f.write(struct.pack('<f', cy))
            f.write(struct.pack('<f', length))

    print(f"[成功] {filename}: 转换了 {num_lines} 条线段，格式 {rows}x{cols} CV_32F")
    return True


if __name__ == "__main__":
    ####################################### 参数 #######################################
    workspace = r"/home/rylynn/Pictures/LinesDetection_Workspace/datasets/Dublin_block3/"

    ####################################### 路径 #######################################
    sparse_model_path = os.path.join(workspace, 'sparse_txt')
    images_path = os.path.join(workspace, 'images')
    output_path = '/home/rylynn/Pictures/LinesDetection_Workspace/output/dublin_block3_md_resize4/L3D++_data_resized/'
    input_lines_path = os.path.join(output_path, 'L3D++_data')
    output_lines_path = os.path.join(output_path, 'L3D++_data_bin')
    os.makedirs(output_lines_path, exist_ok=True)
    
    camerasInfo, points_in_images = load_sparse_model(sparse_model_path, image_scale=1)
    print(f"[INFO] Loaded {len(camerasInfo)} images.")

    for img_id, cam_dict in enumerate(camerasInfo):
        cam_dict = camerasInfo[img_id]
        width = int(cam_dict['width'])
        height = int(cam_dict['height'])
        img_name = cam_dict['img_name'].split('/')[-1]

        input_xml_path = os.path.join(input_lines_path, f'segments_L3D++_{img_id+1}_{width}x{height}.bin')
        output_xml_path = os.path.join(output_lines_path, f'{img_name}.jpg.bin')

        convert_xml_to_opencv_bin(input_xml_path, output_xml_path)


# Matrix city

## convert nerf to colmap and rename images

In [ ]:
import json
import numpy as np
import os
import shutil

def clean_filename(filepath):
    """
    清理文件路径，生成一个唯一的、扁平化的文件名。
    例如：../../train/block_1/0001.png -> train_block_1_0001.png
    """
    parts = os.path.normpath(filepath).split(os.sep)
    # 过滤掉 ".." 和 "train" (如果 "train" 总是根目录)
    cleaned_parts = [p for p in parts if p not in ('..', 'train') and p != '']
    # 将父文件夹和文件名连接起来（例如 block_1_0001.png）
    return "_".join(cleaned_parts)




def convert_nerf_to_colmap(data, sparse_path):
    w = int(data["w"])
    h = int(data["h"])
    fx = data["fl_x"]
    fy = data["fl_y"]
    cx = data.get("cx", w/2)
    cy = data.get("cy", h/2)

    # ==== 2. Write cameras.txt ====
    cameras_filepath = os.path.join(sparse_path, 'cameras.txt')
    with open(cameras_filepath, "w") as f:
        f.write("# Camera list with one line of data per camera:\n")
        f.write("# CAMERA_ID, MODEL, WIDTH, HEIGHT, PARAMS[]\n")
        # Use PINHOLE (fx, fy, cx, cy)
        f.write(f"1 PINHOLE {w} {h} {fx} {fy} {cx} {cy}\n")

    # COLMAP coordinate correction
    R_align = np.array([
        [1, 0, 0],
        [0,-1, 0],
        [0, 0,-1]
    ])

    # ==== 3. Write images.txt ====
    # 记录文件名对应关系
    filename_mapping = {} 
    images_filepath = os.path.join(sparse_path, 'images.txt')
    with open(images_filepath, "w") as f:
        f.write("# Image list with two lines per image:\n")
        f.write("# IMAGE_ID, QW QX QY QZ, TX TY TZ, CAMERA_ID, NAME\n")
        f.write("# POINTS2D[]\n")

        img_id = 1

        for frame in data["frames"]:
            original_filepath = frame["file_path"]
            new_filename = clean_filename(original_filepath)
            filename_mapping[new_filename] = original_filepath.replace("../../train/", "")

            T = np.array(frame["transform_matrix"], dtype=np.float64)  # c2w

            # === Convert c2w → w2c ===
            R = T[:3,:3]
            C = T[:3, 3]
            R_w2c = R.T
            t_w2c = -R.T @ C

            # === Convert to COLMAP camera coordinates ===
            R_colmap = R_align @ R_w2c
            t_colmap = R_align @ t_w2c

            # Convert rotation to quaternion (qw,qx,qy,qz)
            def rot_to_quat(R):
                q = np.empty(4)
                trace = np.trace(R)
                if trace > 0:
                    s = np.sqrt(trace + 1.0) * 2
                    q[0] = 0.25 * s
                    q[1] = (R[2,1] - R[1,2]) / s
                    q[2] = (R[0,2] - R[2,0]) / s
                    q[3] = (R[1,0] - R[0,1]) / s
                else:
                    i = np.argmax(np.diag(R))
                    if i == 0:
                        s = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2]) * 2
                        q[0] = (R[2,1] - R[1,2]) / s
                        q[1] = 0.25 * s
                        q[2] = (R[0,1] + R[1,0]) / s
                        q[3] = (R[0,2] + R[2,0]) / s
                    elif i == 1:
                        s = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2]) * 2
                        q[0] = (R[0,2] - R[2,0]) / s
                        q[1] = (R[0,1] + R[1,0]) / s
                        q[2] = 0.25 * s
                        q[3] = (R[1,2] + R[2,1]) / s
                    else:
                        s = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1]) * 2
                        q[0] = (R[1,0] - R[0,1]) / s
                        q[1] = (R[0,2] + R[2,0]) / s
                        q[2] = (R[1,2] + R[2,1]) / s
                        q[3] = 0.25 * s
                return q / np.linalg.norm(q)

            q = rot_to_quat(R_colmap)

            f.write(f"{img_id} {q[0]} {q[1]} {q[2]} {q[3]} "
                    f"{t_colmap[0]} {t_colmap[1]} {t_colmap[2]} "
                    f"1 {new_filename}\n")
            f.write("\n")   # no 2D-3D matches
            
            if img_id == 1:
                print("--- COLMAP W2C Poses ---")
                print(f"Rotation Matrix R_W2C (COLMAP):\n{R_colmap.round(4)}")
                print(f"Translation T_W2C (COLMAP): {t_colmap.round(4)}")
                print(f"Quaternion [QW, QX, QY, QZ]: {np.array([q[0], q[1], q[2], q[3]]).round(10)}")
            img_id += 1

    # ==== 4. Empty points3D.txt ====
    images_filepath = os.path.join(sparse_path, "points3D.txt")
    with open(images_filepath, "w") as f:
        f.write("# Empty 3d points\n")

    return filename_mapping



image_original_path = "/media/rylynn/data/MatrixCity/"
current_path = "/home/rylynn/Pictures/datasets_3Dline/MatrixCity/block_B"
image_path = os.path.join(current_path, 'images')
os.makedirs(image_path, exist_ok=True)
sparse_path = os.path.join(current_path, 'sparse_original')
os.makedirs(sparse_path, exist_ok=True)
json_path = os.path.join(current_path, 'transforms_train.json')


# ==== 1. Load JSON ====
with open(json_path, "r") as f:
    data = json.load(f)

# 执行转换
filename_mapping = convert_nerf_to_colmap(data, sparse_path)
# 复制图片
for new_filename, original_filepath in filename_mapping.items():
    # 构造原始文件的完整路径
    original_file_fullpath = os.path.join(image_original_path, original_filepath)
    
    # 构造目标文件的完整路径
    target_file_fullpath = os.path.join(image_path, new_filename)
    
    # 确保目标目录存在
    os.makedirs(os.path.dirname(target_file_fullpath), exist_ok=True)
    
    # 复制文件
    shutil.copy2(original_file_fullpath, target_file_fullpath)

print(f"✅ Images from {original_file_fullpath} copied to: {image_path}")

## copy matrix city depth map

In [ ]:
import json
import numpy as np
import os
import shutil

def clean_filename(filepath):
    """
    清理文件路径，生成一个唯一的、扁平化的文件名。
    例如：../../train/block_1/0001.png -> block_1_0001.png
    """
    parts = os.path.normpath(filepath).split(os.sep)
    # 过滤掉 ".." 和 "train" (如果 "train" 总是根目录)
    cleaned_parts = [p for p in parts if p not in ('..', 'train') and p != '']
    # 将父文件夹和文件名连接起来（例如 block_1_0001.png）
    return "_".join(cleaned_parts)




def convert_nerf_to_colmap(data, sparse_path):

    # ==== 3. Write images.txt ====
    # 记录文件名对应关系
    filename_mapping = {} 
    for frame in data["frames"]:
        original_filepath = frame["file_path"]
        new_filename = clean_filename(original_filepath).replace('.png', '.exr')

        depth_folder = original_filepath.replace("../../train/", "").split('/')[0]+"_depth"
        original_depthpath = os.path.join(depth_folder, os.path.basename(original_filepath).replace('.png', '.exr'))
        filename_mapping[new_filename] = original_depthpath
        


    return filename_mapping



image_original_path = "/media/rylynn/data/MatrixCity/"
current_path = "/home/rylynn/Pictures/datasets_3Dline/MatrixCity/block_B"
image_path = os.path.join(current_path, 'depth_maps')
os.makedirs(image_path, exist_ok=True)
sparse_path = os.path.join(current_path, 'sparse_original')
os.makedirs(sparse_path, exist_ok=True)
json_path = os.path.join(current_path, 'transforms_train.json')


# ==== 1. Load JSON ====
with open(json_path, "r") as f:
    data = json.load(f)

# 执行转换
filename_mapping = convert_nerf_to_colmap(data, sparse_path)
# 复制深度图
for new_filename, original_filepath in filename_mapping.items():
    # 构造原始文件的完整路径
    original_file_fullpath = os.path.join(image_original_path, original_filepath)
    
    # 构造目标文件的完整路径
    target_file_fullpath = os.path.join(image_path, new_filename)
    
    # 确保目标目录存在
    os.makedirs(os.path.dirname(target_file_fullpath), exist_ok=True)
    
    # 复制文件
    shutil.copy2(original_file_fullpath, target_file_fullpath)

print(f"✅ Images from {original_file_fullpath} copied to: {image_path}")